In [ ]:
## 필요한 파이썬 라이브러리를 임포트 합니다.
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

KeyboardInterrupt: 

torchvision에서 제공하는 데이터셋: https://pytorch.org/vision/main/datasets.html

데이터 전처리 및 증강 관련 documents: https://pytorch.org/vision/0.9/transforms.html, https://github.com/rasbt/deeplearning-models/blob/master/pytorch_ipynb/mechanics/torchvision-transform-examples.ipynb

## 1. 기본 CNN 구조의 정의 및 활용

In [ ]:
# 데이터 전처리 단계입니다. 이미지 데이터를 처리할 때는 다음의 단계가 일반적으로 필요합니다.
transform = transforms.Compose([
    # 이미지를 PyTorch 텐서로 변환합니다.
    transforms.ToTensor(),

    # 정규화 과정을 거쳐 이미지 데이터의 범위를 [-1, 1]로 조정합니다.
    # 여기서는 RGB 각 채널의 평균을 0.5, 표준편차를 0.5로 설정하여 정규화를 수행합니다.
    # 이는 데이터를 모델이 더 잘 학습할 수 있는 형태로 만들어 줍니다.
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# CIFAR10 학습 데이터셋을 로드합니다.
# root='./data': 데이터가 저장될 경로입니다.
# train=True: 학습용 데이터셋을 불러옵니다.
# download=True: 인터넷에서 데이터를 다운로드하고 root에 지정된 경로에 저장합니다.
# transform=transform: 위에서 정의한 전처리 방식을 적용합니다.
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# 데이터 로더를 정의합니다.
# 이 객체는 데이터를 배치 크기에 맞게 묶어주고, 학습 데이터를 섞어서 모델의 일반화 능력을 향상시킵니다.
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4, shuffle=True)

# CIFAR10 테스트 데이터셋을 로드하는 과정은 학습 데이터셋 로드와 유사합니다.
# 단, train=False로 설정하여 테스트 데이터셋을 불러옵니다.
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# 테스트 데이터 로더를 정의합니다.
# 여기서는 shuffle=False로 설정하여 데이터를 섞지 않습니다.
# 테스트 데이터셋을 순서대로 사용하여 모델의 성능을 평가할 때 일관된 결과를 얻기 위함입니다.
testloader = torch.utils.data.DataLoader(testset, batch_size=10, shuffle=False)

# CIFAR10 데이터셋의 클래스 목록을 정의합니다.
# 이는 레이블에 해당하는 실제 객체의 이름을 나타냅니다.
# 예를 들어, 'plane'은 레이블 0에, 'car'는 레이블 1에 해당합니다.
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')


In [ ]:
# CNN 클래스는 PyTorch의 nn.Module을 상속받아 새로운 신경망 아키텍처를 정의합니다.
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()  # nn.Module의 __init__ 함수를 상속받습니다.
        # 첫 번째 합성곱(convolution) 레이어를 정의합니다.
        # 이 레이어는 3개의 입력 채널(RGB 이미지)을 받아, 6개의 출력 채널을 생성하며, 5x5 커널을 사용합니다.
        self.conv1 = nn.Conv2d(3, 6, 5)
        # 맥스 풀링 레이어를 정의합니다.
        # 이는 2x2 윈도우를 사용하여 특성 맵(feature map)의 크기를 1/4으로 줄입니다.
        self.pool = nn.MaxPool2d(2, 2)
        # 두 번째 합성곱 레이어를 정의합니다.
        # 6개의 입력 채널(이전 레이어의 출력)을 받아, 16개의 출력 채널을 생성하며, 5x5 커널을 사용합니다.
        self.conv2 = nn.Conv2d(6, 16, 5)
        # 첫 번째 완전 연결(fully connected) 레이어입니다.
        # 16*5*5 크기의 입력을 받아, 120 크기의 출력을 생성합니다.
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        # 두 번째 완전 연결 레이어입니다.
        # 120 크기의 입력을 받아, 84 크기의 출력을 생성합니다.
        self.fc2 = nn.Linear(120, 84)
        # 세 번째 완전 연결 레이어입니다.
        # 84 크기의 입력을 받아, 최종적으로 10개의 클래스를 분류하기 위한 10 크기의 출력을 생성합니다.
        self.fc3 = nn.Linear(84, 10)

    # 신경망의 순전파 함수를 정의합니다.
    # 이 함수는 신경망에 입력 데이터를 통과시키고 최종 출력을 반환합니다.
    def forward(self, x):
        # 첫 번째 합성곱 레이어를 통과시킨 후, 활성화 함수 ReLU를 적용하고, 풀링합니다.
        x = self.pool(F.relu(self.conv1(x)))
        # 두 번째 합성곱 레이어를 통과시킨 후, 동일하게 ReLU와 풀링을 적용합니다.
        x = self.pool(F.relu(self.conv2(x)))
        # 다차원의 데이터를 1차원으로 펼칩니다(flatten).
        x = x.view(-1, 16 * 5 * 5)
        # 첫 번째 완전 연결 레이어를 통과시킨 후, ReLU를 적용합니다.
        x = F.relu(self.fc1(x))
        # 두 번째 완전 연결 레이어를 통과시킨 후, ReLU를 적용합니다.
        x = F.relu(self.fc2(x))
        # 마지막 완전 연결 레이어를 통과시킵니다.
        # 마지막 레이어에서는 활성화 함수를 적용하지 않습니다.
        x = self.fc3(x)
        return x

# 정의한 CNN 모델의 인스턴스를 생성합니다.
net = CNN()

In [ ]:
## 모델의 손실 함수와 옵티마이저를 정의합니다.

# 교차 엔트로피 손실 함수를 사용합니다.
# 이 함수는 분류 문제에서 일반적으로 사용되며, 예측과 실제 레이블 간의 차이를 측정합니다.
criterion = nn.CrossEntropyLoss()

# 확률적 경사 하강법(SGD) 옵티마이저를 사용하여 모델의 파라미터를 최적화합니다.
# lr=0.001: 학습률(learning rate)을 설정합니다. 이 값은 파라미터 업데이트의 크기를 결정합니다.
# momentum=0.9: 모멘텀 값을 설정하여 지난 그래디언트의 일부를 현재 업데이트에 추가함으로써,
# 학습 과정의 안정성과 속도를 높이는 데 도움을 줍니다.
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
# 학습 과정을 시작합니다. 여기서는 총 10회의 에폭(epoch) 동안 학습을 수행합니다.
# 에폭은 전체 데이터셋을 한 번 학습하는 단위입니다.
for epoch in range(10):

    running_loss = 0.0  # 현재 에폭에서의 손실을 추적하기 위한 변수를 초기화합니다.

    # 학습 데이터셋을 순회하는 루프를 시작합니다.
    # enumerate를 사용하여 반복 횟수와 데이터를 함께 반환받습니다.
    for i, data in enumerate(trainloader, 0):
        # 각 미니배치에서 입력 이미지와 해당 레이블을 로드합니다.
        inputs, labels = data

        # 옵티마이저에 저장된 기존의 그래디언트 정보를 초기화합니다.
        optimizer.zero_grad()

        # 순전파 단계: 모델에 입력을 넣고 출력을 얻습니다.
        outputs = net(inputs)
        # 손실 함수를 사용하여 예측 오류를 계산합니다.
        loss = criterion(outputs, labels)
        # 역전파 단계: 손실에 대한 모델의 파라미터의 그래디언트를 계산합니다.
        loss.backward()
        # 옵티마이저를 사용하여 파라미터를 업데이트합니다.
        optimizer.step()

        # 출력된 손실을 running_loss에 누적합니다.
        running_loss += loss.item()
        # 매 2000 미니배치마다 평균 손실을 계산하여 출력합니다.
        if i % 2000 == 1999:
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000))
            # 다음 2000 미니배치의 손실 계산을 위해 running_loss를 다시 0으로 설정합니다.
            running_loss = 0.0

# 모든 학습 에폭이 완료되었음을 알립니다.
print('Finished Training')

In [ ]:
# 정확도를 계산하기 위한 초기값 설정
correct = 0  # 올바르게 예측된 데이터의 수
total = 0    # 전체 데이터의 수

# 평가 과정에서는 모델을 업데이트하지 않으므로, 그래디언트 계산을 비활성화합니다.
with torch.no_grad():
    # 테스트 데이터셋을 순회합니다.
    for data in testloader:
        # 테스트 데이터셋의 이미지와 정답 레이블을 로드합니다.
        images, labels = data
        # 학습된 모델을 사용하여 이미지에 대한 예측을 수행합니다.
        outputs = net(images)
        # 가장 높은 값(가장 확신 있는 예측)을 가진 인덱스를 가져옵니다.
        _, predicted = torch.max(outputs.data, 1)
        # 전체 데이터의 수를 업데이트합니다.
        total += labels.size(0)
        # 올바르게 예측된 데이터의 수를 계산합니다.
        correct += (predicted == labels).sum().item()

# 전체 테스트 데이터에 대한 모델의 정확도를 출력합니다.
print('Accuracy of the network on the 10000 test images: %d %%' % (
    100 * correct / total))

In [ ]:
import numpy as np

i = 5 # i = 4
im = images[i].detach().cpu().numpy()
im = np.transpose(im, [1, 2, 0])

plt.figure(figsize=(3, 3))
plt.imshow(im)
plt.title("Truth: %s, Predicted: %s" %(classes[labels[i]], classes[predicted[i]]))
plt.show()

## 2. 최신 CNN 알고리즘 로드하여 활용하기 (torchvision.models)
(https://pytorch.org/vision/stable/models.html)

In [ ]:
# 사전 학습된(pretrained) 모델을 로드하는 torchvision 라이브러리를 불러옵니다.
from torchvision import models

## VGG-16 모델을 로드합니다.
# 이 모델은 이미 대규모 이미지 데이터셋에서 학습이 완료된 상태입니다(pretrained=True).
vgg16 = models.vgg16(pretrained=True)
# CIFAR-10은 10개의 클래스를 가지므로, VGG-16 모델의 마지막 분류기(classifier) 계층을
# 10개의 출력을 가지는 새로운 계층으로 교체합니다.
vgg16.classifier[6] = torch.nn.Linear(vgg16.classifier[6].in_features, 10)

## ResNet-50 모델을 로드합니다.
# 이 또한 사전 학습된 상태로 로드합니다.
resnet50 = models.resnet50(pretrained=True)
# 마찬가지로 마지막 완전 연결층(fc)을 10개 클래스를 분류할 수 있는 새로운 계층으로 교체합니다.
resnet50.fc = torch.nn.Linear(resnet50.fc.in_features, 10)

## Inception v3 모델을 로드합니다.
inception_v3 = models.inception_v3(pretrained=True)
# Inception 모델은 보조 출력(auxiliary output)이 있으므로 이를 처리합니다.
# 보조 출력을 위한 완전 연결층의 입력 특징 수를 가져옵니다.
in_features = inception_v3.AuxLogits.fc.in_features
# 보조 출력의 완전 연결층을 새로운 계층으로 교체합니다.
inception_v3.AuxLogits.fc = torch.nn.Linear(in_features, 10)
# 주 출력(primary output)에 대해서도 동일하게 처리합니다.
in_features = inception_v3.fc.in_features
inception_v3.fc = torch.nn.Linear(in_features, 10)

In [ ]:
## 손실 함수와 옵티마이저를 정의합니다.
# 교차 엔트로피 손실 함수는 분류 작업에 일반적으로 사용됩니다.
criterion = nn.CrossEntropyLoss()
# 확률적 경사 하강법(SGD) 옵티마이저를 사용하며, 학습률(lr)과 모멘텀을 설정합니다.
optimizer = optim.SGD(resnet50.parameters(), lr=0.001, momentum=0.9)

In [ ]:
## 학습을 시작합니다.
# 전체 데이터셋을 10번 반복하여 모델을 학습합니다.
for epoch in range(10):

    running_loss = 0.0  # 에폭마다 손실을 추적하기 위한 변수를 초기화합니다.
    # 학습 데이터로더를 통해 미니배치를 순회합니다.
    for i, data in enumerate(trainloader, 0):
        # 입력 데이터와 레이블을 가져옵니다.
        inputs, labels = data

        # 옵티마이저를 초기화하여 이전 반복에서의 그래디언트를 지웁니다.
        optimizer.zero_grad()

        # 순전파를 수행하고 손실을 계산한 후, 역전파를 통해 그래디언트를 계산하고 파라미터를 업데이트합니다.
        outputs = resnet50(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # 손실을 누적하여 통계를 출력합니다.
        running_loss += loss.item()
        # 2000 미니배치마다 평균 손실을 출력합니다.
        if i % 2000 == 1999:
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000))
            running_loss = 0.0

# 학습이 끝나면 완료 메시지를 출력합니다.
print('Finished Training')

In [ ]:
## 평가 과정입니다.
# 맞게 예측한 샘플 수와 전체 샘플 수를 초기화합니다.
correct = 0
total = 0
# 모델을 평가할 때는 그래디언트를 계산할 필요가 없습니다.
with torch.no_grad():
    # 테스트 데이터로더를 통해 데이터를 순회합니다.
    for data in testloader:
        images, labels = data
        # 모델을 사용하여 예측을 수행합니다.
        outputs = resnet50(images)
        # 예측 결과 중 가장 높은 값을 선택합니다.
        _, predicted = torch.max(outputs.data, 1)
        # 전체 샘플 수를 업데이트합니다.
        total += labels.size(0)
        # 맞게 예측한 샘플 수를 계산합니다.
        correct += (predicted == labels).sum().item()

# 전체 테스트 데이터에 대한 정확도를 계산하여 출력합니다.
print('Accuracy of the network on the 10000 test images: %d %%' % (
    100 * correct / total))